# Stage 7-2. Binary 전체 실험 — MLP + CNN 비교

이 노트북은 MNIST digit을 짝수/홀수 두 범주로 분류하는 전체 실험 파이프라인을 실습한다.

| 항목 | 값 |
|---|---|
| 과제 | 짝수(0) vs 홀수(1) 이진 분류 |
| target 변환 | `label % 2` → shape `(1,)` |
| output_dim | 1 |
| loss | binary_cross_entropy (sigmoid 내부 처리) |
| metric | binary_accuracy |
| prediction_mode | threshold (sigmoid ≥ 0.5) |

**학습 목표**
1. 같은 `Experiment` 구조에서 `task="binary"`만 바꿔 이진 분류를 실행한다.
2. 이진 분류의 target 변환, loss, metric, prediction 후처리를 확인한다.
3. MLP vs CNN 학습 곡선과 예측 grid를 비교한다.
4. 저장된 checkpoint로 평가를 재현한다.
5. Multiclass vs Binary 차이를 task 규약 수준에서 정리한다.

## 0. 환경 설정

In [ ]:
import sys
sys.path.insert(0, "../..")

import numpy as np
import matplotlib.pyplot as plt
import os

try:
    import cupy
    _BACKEND = f"CuPy {cupy.__version__}"
except ImportError:
    _BACKEND = "NumPy (CuPy 미설치 — fallback)"

print(f"백엔드: {_BACKEND}")

from src.core.experiment import Experiment
from src.core import checkpoints
from src.core.evaluator import Evaluator
from src.data.mnist import MnistDataset
from src.data.dataloader import DataLoader
from src.models.mlp import MLP
from src.models.cnn import CNN
from src.task import get_task_spec

DATASET_DIR = "/mnt/d/datasets/mnist"
OUTPUT_DIR  = "../../outputs/binary"
TASK        = "binary"

print(f"\ntask: {TASK}")
spec = get_task_spec(TASK)
print(f"  output_dim     : {spec['output_dim']}")
print(f"  prediction_mode: {spec['prediction_mode']}")

## 1. Binary task 규약 확인

In [ ]:
# target 변환 확인
ds = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)

print("[ Binary target 변환 확인 (처음 10개 샘플) ]")
print(f"  {'원본 digit':>12} {'binary target':>14} {'의미':>8}")
print("  " + "-" * 38)

# test split의 원본 label을 load_mnist로 확인
from src.data.mnist import load_mnist
_, orig_labels = load_mnist("test", dataset_dir=DATASET_DIR)

for i in range(10):
    _, target = ds[i]
    digit = int(orig_labels[i])
    bt    = int(target[0])
    meaning = "홀수" if bt == 1 else "짝수"
    print(f"  {digit:>12} {bt:>14} {meaning:>8}")

print()
print(f"target shape : {ds[0][1].shape}")
print(f"target dtype : {ds[0][1].dtype}")
print(f"target 변환  : label % 2  (홀수=1, 짝수=0)")

In [ ]:
# class 분포 시각화
targets = np.array([int(ds[i][1][0]) for i in range(len(ds))])
counts  = np.bincount(targets, minlength=2)

fig, ax = plt.subplots(figsize=(5, 3.5))
bars = ax.bar(["짝수 (0)", "홀수 (1)"], counts, color=["steelblue", "#E87B4C"],
              edgecolor="gray", alpha=0.85)
for bar, cnt in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f"{cnt:,}", ha="center", fontsize=11, fontweight="bold")
ax.set_title("Binary target 분포 (test split)", fontsize=11, fontweight="bold")
ax.set_ylabel("샘플 수")
ax.set_ylim(0, max(counts) * 1.18)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"짝수: {counts[0]:,}개 ({counts[0]/len(ds):.1%})")
print(f"홀수: {counts[1]:,}개 ({counts[1]/len(ds):.1%})")
print(f"→ 두 클래스가 거의 균등하게 분포 (random baseline ≈ 50%)")

## 2. MLP 학습

In [ ]:
config_mlp = {
    "dataset_dir": DATASET_DIR,
    "task":        TASK,
    "model":       "mlp",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  10,
    "lr":          0.01,
}

exp_mlp = Experiment(config_mlp)
print(f"MLP 파라미터 수: {sum(p.size for p in exp_mlp.model.params):,}")
print("학습 시작 (10 epochs)...")
logs_mlp = exp_mlp.run()

print()
print(f"{'epoch':>5} | {'train loss':>10} {'train acc':>12} | {'test loss':>10} {'test acc':>12}")
print("-" * 58)
for log in logs_mlp:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>12.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>12.4f}")

## 3. CNN 학습

In [ ]:
config_cnn = {
    "dataset_dir": DATASET_DIR,
    "task":        TASK,
    "model":       "cnn",
    "seed":        42,
    "batch_size":  64,
    "num_epochs":  10,
    "lr":          0.01,
}

exp_cnn = Experiment(config_cnn)
print(f"CNN 파라미터 수: {sum(p.size for p in exp_cnn.model.params):,}")
print("학습 시작 (10 epochs)...")
logs_cnn = exp_cnn.run()

print()
print(f"{'epoch':>5} | {'train loss':>10} {'train acc':>12} | {'test loss':>10} {'test acc':>12}")
print("-" * 58)
for log in logs_cnn:
    print(f"{log['epoch']:>5} | "
          f"{log['train']['loss']:>10.4f} {log['train']['metric']:>12.4f} | "
          f"{log['test']['loss']:>10.4f} {log['test']['metric']:>12.4f}")

## 4. 학습 곡선 비교

In [ ]:
epochs = list(range(1, 11))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
fig.suptitle("Binary — MLP vs CNN (10 epochs, lr=0.01, batch=64)",
             fontsize=12, fontweight="bold")

for ax, key, title in zip(axes, ["loss", "metric"], ["Binary Cross-Entropy Loss", "Binary Accuracy"]):
    ax.plot(epochs, [l["train"][key] for l in logs_mlp], color="steelblue", linewidth=2,
            marker="o", label="MLP train")
    ax.plot(epochs, [l["test"][key]  for l in logs_mlp], color="steelblue", linewidth=2,
            marker="o", linestyle="--", label="MLP test")
    ax.plot(epochs, [l["train"][key] for l in logs_cnn], color="#E87B4C",   linewidth=2,
            marker="s", label="CNN train")
    ax.plot(epochs, [l["test"][key]  for l in logs_cnn], color="#E87B4C",   linewidth=2,
            marker="s", linestyle="--", label="CNN test")
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("epoch")
    ax.set_ylabel(title.lower())
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

axes[1].set_ylim(0, 1)
axes[1].axhline(0.5, color="gray", linestyle=":", linewidth=1.5, label="random baseline")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

## 5. Prediction 후처리 확인

In [ ]:
# sigmoid threshold prediction 확인
from src.nn.activations import sigmoid

test_ds = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
imgs    = np.stack([test_ds[i][0] for i in range(8)])

result = exp_mlp.predictor.predict(imgs)
logits = result["logits"]
probs  = sigmoid(logits)
preds  = result["predictions"]

print("[ Binary 예측 후처리 — sigmoid threshold ]")
print(f"  {'샘플':>4} {'logit':>8} {'sigmoid':>9} {'pred':>6} {'의미':>6}")
print("  " + "-" * 38)
for i in range(8):
    meaning = "홀수" if int(preds[i]) == 1 else "짝수"
    print(f"  {i:>4} {float(logits[i,0]):>8.3f} {float(probs[i,0]):>9.4f} {int(preds[i]):>6} {meaning:>6}")

print()
print("규칙: sigmoid(logit) ≥ 0.5 → 홀수(1), < 0.5 → 짝수(0)")

In [ ]:
# 예측 grid 비교
N_SHOW = 16
imgs   = np.stack([test_ds[i][0] for i in range(N_SHOW)])
trues  = np.array([int(test_ds[i][1][0]) for i in range(N_SHOW)])

preds_mlp = exp_mlp.predictor.predict(imgs)["predictions"].flatten().astype(int)
preds_cnn = exp_cnn.predictor.predict(imgs)["predictions"].flatten().astype(int)

label_map = {0: "짝", 1: "홀"}

fig, axes = plt.subplots(2, 8, figsize=(14, 5))
fig.suptitle("Binary 예측 grid — MLP(위) vs CNN(아래) (10 epochs)",
             fontsize=11, fontweight="bold")

for col in range(8):
    for row, (preds, label) in enumerate([(preds_mlp, "MLP"), (preds_cnn, "CNN")]):
        ax = axes[row][col]
        ax.imshow(imgs[col].reshape(28, 28), cmap="gray")
        t, p = trues[col], preds[col]
        color = "green" if t == p else "red"
        ax.set_title(f"T:{label_map[t]} P:{label_map[p]}", fontsize=7, color=color)
        ax.axis("off")
        if col == 0:
            ax.set_ylabel(label, fontsize=9)

plt.tight_layout()
plt.show()

print(f"MLP 정답률 ({N_SHOW}개): {(trues == preds_mlp).mean():.0%}")
print(f"CNN 정답률 ({N_SHOW}개): {(trues == preds_cnn).mean():.0%}")

## 6. 정량 비교 및 checkpoint 재평가

In [ ]:
mlp_params = sum(p.size for p in exp_mlp.model.params)
cnn_params = sum(p.size for p in exp_cnn.model.params)
mlp_final  = logs_mlp[-1]["test"]
cnn_final  = logs_cnn[-1]["test"]

print("[ Binary 최종 결과 비교 ]")
print()
print(f"{'항목':<22} {'MLP':>12} {'CNN':>12}")
print("-" * 48)
print(f"{'파라미터 수':<22} {mlp_params:>12,} {cnn_params:>12,}")
print(f"{'test loss':<22} {mlp_final['loss']:>12.4f} {cnn_final['loss']:>12.4f}")
print(f"{'test accuracy':<22} {mlp_final['metric']:>12.4f} {cnn_final['metric']:>12.4f}")
print(f"{'accuracy 향상 (CNN-MLP)':<22} {'':>12} {cnn_final['metric'] - mlp_final['metric']:>+12.4f}")

print()
print("[ Phase 7.2 기준 결과 (저장된 checkpoint 평가) ]")
print(f"  MLP: loss=0.2923, accuracy=0.8814")
print(f"  CNN: loss=0.1780, accuracy=0.9347")

# 저장된 checkpoint 재평가
test_ds     = MnistDataset("test", TASK, dataset_dir=DATASET_DIR)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)
spec        = get_task_spec(TASK)

print()
print("[ 저장된 checkpoint 평가 ]")
for model_name, ModelClass in [("mlp", MLP), ("cnn", CNN)]:
    ck_path = os.path.join(OUTPUT_DIR, model_name, "model")
    if os.path.isfile(ck_path + ".npz"):
        model = ModelClass(task=TASK, seed=42)
        checkpoints.load(model, ck_path)
        evaluator = Evaluator(model, spec)
        result = evaluator.evaluate(test_loader)
        print(f"  {model_name.upper()}: loss={result['loss']:.4f}, "
              f"accuracy={result['metric']:.4f}, samples={result['num_samples']}")
    else:
        print(f"  {model_name.upper()}: checkpoint 없음")

## 7. 저장된 결과 이미지 확인

In [ ]:
from PIL import Image
import warnings
warnings.filterwarnings("ignore")

fig, axes = plt.subplots(2, 2, figsize=(14, 7))
fig.suptitle("outputs/binary/ — 저장된 결과 이미지", fontsize=12, fontweight="bold")

for row, model_name in enumerate(["mlp", "cnn"]):
    for col, fname in enumerate(["training_log.png", "predictions.png"]):
        ax = axes[row][col]
        fpath = os.path.join(OUTPUT_DIR, model_name, fname)
        if os.path.isfile(fpath):
            ax.imshow(np.asarray(Image.open(fpath)))
            ax.set_title(f"{model_name.upper()} — {fname}", fontsize=9)
        else:
            ax.text(0.5, 0.5, "파일 없음", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(f"{model_name.upper()} — {fname}", fontsize=9)
        ax.axis("off")

plt.tight_layout()
plt.show()

## 8. CLI 실행 명령

In [ ]:
print("[ Binary 실험 CLI 실행 명령 ]")
print()
for model, env in [("mlp", "numpy_py311"), ("cnn", "cupy_py311_cuda121")]:
    print(f"# {model.upper()} ({env})")
    print(f"PYTHONPATH=. conda run -n {env} python scripts/train.py \\\ ")
    print(f"    --task binary --model {model} \\\ ")
    print(f"    --checkpoint outputs/binary/{model}/model.npz")
    print()
    print(f"PYTHONPATH=. conda run -n {env} python scripts/evaluate.py \\\ ")
    print(f"    --task binary --model {model} \\\ ")
    print(f"    --checkpoint outputs/binary/{model}/model.npz")
    print()

## 9. 정리

**Binary 과제 규약**

| 항목 | 값 |
|---|---|
| target 변환 | `label % 2` → 홀수=1, 짝수=0 |
| output_dim | 1 (scalar logit) |
| loss | binary_cross_entropy (내부에서 sigmoid 처리) |
| metric | binary_accuracy |
| prediction_mode | threshold: `sigmoid(logit) ≥ 0.5` |

**Multiclass vs Binary 차이**

| 항목 | Multiclass | Binary |
|---|---|---|
| target | one-hot `(10,)` | scalar `(1,)` |
| output_dim | 10 | 1 |
| loss | cross_entropy | binary_cross_entropy |
| activation | softmax (loss 내부) | sigmoid (loss 내부) |
| prediction | argmax | threshold ≥ 0.5 |

**실험 결과 요약 (Phase 7.2 기준)**

| 모델 | test loss | test accuracy |
|---|---|---|
| MLP | 0.2923 | 0.8814 |
| CNN | 0.1780 | 0.9347 |

Binary 과제는 Multiclass에 비해 단순한 문제이지만, CNN의 공간 특징 추출 능력이 동일하게 작동하여 MLP 대비 높은 정확도를 보인다.